# Tutorial 5 · Security and commitment

**Difficulty: intermediate–advanced · ~8–10 min read**

[Tutorial 2](02-dc-opf-and-n1.ipynb) found the cheapest dispatch and then asked whether it
survives any single outage — and on the rated case14 it does not. This tutorial is the rest of
that story: what a planner does about it. We rate the lines twice, screen single and double
outages, find the cheapest dispatch that is *secure* under three different definitions of the
word, and finish with a horizon where the question is not only how much each unit makes but
whether it is switched on at all, and how much of it is held back as reserve.

Everything here is the DC model, and every claim is confirmed by a real re-solve that travels
with the result. The decision records are ADR-016 and ADR-019 to ADR-025 in
[Design decisions](../design/decisions.md).

In [1]:
from mambo_power import cases, contingency, opf
from mambo_power.market.multiperiod import MarketMultiperiodOptions, solve_multiperiod
from mambo_power.model import Bus, Generator, Load, Network, Period, PolynomialCost, Scenario

net = cases.load("derived/case14_rated")
print(len(net.branches), "branches,", sum(br.rating_mva is not None for br in net.branches), "rated")

20 branches, 20 rated


## Two ratings per line

`derived/case14_rated` is IEEE case14 with a thermal rating on every branch (the bundled
`case14` has none — MATPOWER's convention for "unlimited"). A line can usually carry more than
its continuous rating for the minutes after a contingency, until operators act, and PSS/E and
MATPOWER both carry that second number (`RATEC`). In this package it is
`Branch.rating_emergency_mva`: the limit a *post-contingency* flow is held to, where the normal
`rating_mva` limits the intact network. Nothing base-case ever reads it.

Let's give every rated branch 30 % of emergency headroom and see what it does to the N-1 screen.

In [2]:
relaxed = net.model_copy(deep=True)
for br in relaxed.branches:
    if br.rating_mva is not None:
        br.rating_emergency_mva = 1.3 * br.rating_mva

for label, network in (("normal rating", net), ("30 % emergency rating", relaxed)):
    result = contingency.n1(network)
    violating = sum(o.confirmed_violating for o in result.outages)
    print(f"{label:22s} {len(result.outages):2d} branch outages flagged, {violating:2d} confirmed violating")

normal rating          11 branch outages flagged, 11 confirmed violating
30 % emergency rating   6 branch outages flagged,  6 confirmed violating


Note that *every* outage flagged by the screen was confirmed by a DC re-solve with that branch
out: `confirmed_violating` is the re-solve's verdict, not the estimate's. That is the pattern
every security tool here follows — a fast linear screen says where to look, a real solve says
what is true, and the result carries both.

## Double outages

Single outages are the usual criterion; planners also ask about pairs. `contingency.n2` screens
every pair of branch outages at once. Two outages are not two N-1 estimates added up (each one's
flow spills over the other, now-missing, branch), but the DC model is linear and the joint effect
is a closed form — the generalised LODF — so the screen is one pass of linear algebra over all
pairs. Some pairs cannot be screened at all: removing both branches disconnects the network, and
the screen detects that from a vanishing determinant rather than returning a meaningless number.

In [3]:
pairs = contingency.n2(net)
print(pairs.n_pairs_screened, "pairs screened;", len(pairs.outages), "flagged and confirmed by re-solve")
print(len(pairs.islanding_pairs), "pairs disconnect the network, e.g.", pairs.islanding_pairs[:2])
worst = max(
    pairs.outages,
    key=lambda o: max(f.confirmed_flow_mw / f.rating_mva for f in o.flagged_branches),
)
flag = max(worst.flagged_branches, key=lambda f: f.confirmed_flow_mw / f.rating_mva)
print(f"worst: {worst.outage_branch_ids} out puts {flag.confirmed_flow_mw:.1f} MW on {flag.branch_id} rated {flag.rating_mva:.1f}")

171 pairs screened; 147 flagged and confirmed by re-solve
8 pairs disconnect the network, e.g. [['branch-1', 'branch-2'], ['branch-3', 'branch-6']]
worst: ['branch-3', 'branch-7'] out puts 94.2 MW on branch-6 rated 30.0


## The cheapest dispatch that is secure — three ways

`opf.solve_scopf` is the security-constrained DC-OPF: the cheapest dispatch such that no single
outage overloads anything. It does not enumerate every (outage, branch) pair up front — on
case300 that would be ~170 000 rows, almost all slack — but *generates* them: solve, run the
N-1 screen on the dispatch, add a row for each flagged pair, solve again, until the screen
flags nothing new. Each row is a post-contingency flow limit, linear in the dispatch under the
DC model. The secure dispatch is then confirmed by running the real `contingency.n1` on it.

Three definitions of "secure" are on offer, and they bracket each other:

1. **Preventive at the normal rating** — one dispatch, every post-contingency flow inside the
   normal rating. The most constrained.
2. **Preventive at the emergency rating** — the same, but the immediate post-contingency flow
   only has to fit the emergency rating. The least constrained, and it promises nothing about
   getting back under the normal rating.
3. **Corrective** — the immediate flow fits the emergency rating *and* a redispatch exists,
   each unit within its ramp of the base dispatch, that brings every flow back under the normal
   rating. In between, and the one operators actually run.

In [4]:
corrective_net = relaxed.model_copy(deep=True)
for gen in corrective_net.generators:
    gen.ramp_up_mw = gen.ramp_down_mw = 20.0  # what a unit can move in the emergency window

plain = opf.solve_dc_opf(net)
answers = {
    "unsecured (plain DC-OPF)": plain.objective_cost,
    "preventive, normal rating": opf.solve_scopf(net).objective_cost,
    "preventive, emergency rating": opf.solve_scopf(relaxed).objective_cost,
    "corrective, 20 MW ramps": opf.solve_scopf(
        corrective_net, opf.ScopfOptions(corrective=True)
    ).objective_cost,
}
for label, cost in answers.items():
    print(f"{label:30s} {cost:9.2f} $/h  (+{cost - plain.objective_cost:7.2f})")

unsecured (plain DC-OPF)         7642.59 $/h  (+   0.00)
preventive, normal rating        8651.92 $/h  (+1009.33)
preventive, emergency rating     8314.11 $/h  (+ 671.52)
corrective, 20 MW ramps          8447.25 $/h  (+ 804.66)


The corrective answer sits between the two preventive ones, as it must: it is the
emergency-rating problem with one more requirement. What that requirement buys is a dispatch
that can be *repaired* — and the result carries the repair for every outage that needs one,
each confirmed by a DC re-solve with the outage applied and the corrected dispatch written in.

In [5]:
corrected = opf.solve_scopf(corrective_net, opf.ScopfOptions(corrective=True))
print(corrected.status, "secure:", corrected.secure, "corrected_secure:", corrected.corrected_secure)
print(len(corrected.corrections), "outages need a correction; the largest:")
largest = max(corrected.corrections, key=lambda c: max(abs(r.delta_mw) for r in c.redispatch))
moves = ", ".join(f"{r.id} {r.delta_mw:+.1f}" for r in largest.redispatch if abs(r.delta_mw) > 1e-6)
print(f"  {largest.outage_kind} {largest.outage_id} out -> {moves}")
print(f"  re-solved loading after the correction: {100 * largest.confirmed_max_loading:.0f} % of the normal rating")

Optimal secure: True corrected_secure: True
18 outages need a correction; the largest:
  generator gen-2 out -> gen-1 +20.0, gen-2 -56.3, gen-3 +20.0, gen-4 +20.0, gen-5 -3.7
  re-solved loading after the correction: 75 % of the normal rating


### Securing against pairs too

The N-2 screen's closed form is linear in the dispatch as well, so the same constraint
generation takes pair rows: `ScopfOptions(branch_pairs=True)`. At the normal ratings no
dispatch of case14_rated survives every pair — some pair leaves a load with a single path rated
below it — and the result says so rather than returning something. With the 30 % emergency
headroom it can be done, for a little more than N-1 alone.

In [6]:
at_normal = opf.solve_scopf(net, opf.ScopfOptions(branch_pairs=True))
print("N-2 at the normal rating:", at_normal.status)
n2_secure = opf.solve_scopf(relaxed, opf.ScopfOptions(branch_pairs=True))
assert n2_secure.n2 is not None
print(
    f"N-2 at the emergency rating: {n2_secure.status}, secure = {n2_secure.secure}, "
    f"{n2_secure.objective_cost:.2f} $/h; {len(n2_secure.constraints)} rows, "
    f"{sum(c.outage_kind == 'pair' for c in n2_secure.constraints)} of them pair rows; "
    f"the confirming N-2 screen flags {len(n2_secure.n2.outages)} pairs"
)

N-2 at the normal rating: Infeasible
N-2 at the emergency rating: Optimal, secure = True, 8456.76 $/h; 224 rows, 210 of them pair rows; the confirming N-2 screen flags 0 pairs


## Commitment and reserve over a horizon

Everything so far dispatched units that were already on. Over a day the question is also *which*
units to run, and that is a different kind of problem: a unit with a minimum output cannot be
dispatched to zero, so it has to be switched off, which costs a start-up to undo. The
multiperiod market clears this as a mixed-integer program when asked
(`MarketMultiperiodOptions(unit_commitment=True)`), with start-up and no-load costs, minimum
up and down times, and — the last piece — a **spinning reserve** requirement per period: capacity
the committed fleet must hold *unused*, priced by its own dual.

One bus, two units, three periods. The `peaker` is dear and on; the `baseload` unit is cheap,
has a 40 MW minimum and an $8000 start-up, and is off before the horizon.

In [7]:
def unit(gen_id, p_mw, p_min, p_max, marginal, startup=0.0, reserve_max=None):
    return Generator(
        id=gen_id, bus="b1", p_mw=p_mw, q_mvar=0.0, p_min_mw=p_min, p_max_mw=p_max,
        q_min_mvar=0.0, q_max_mvar=0.0, v_set_pu=1.0, reserve_max_mw=reserve_max,
        cost=PolynomialCost(coefficients=[marginal, 0.0], startup=startup),
    )  # fmt: skip


def scenario(reserve_mw=None):
    network = Network(
        base_mva=100.0,
        buses=[Bus(id="b1", base_kv=138.0, type="slack")],
        generators=[
            unit("peaker", 10.0, 0.0, 100.0, 50.0, reserve_max=20.0),
            unit("baseload", 0.0, 40.0, 200.0, 10.0, startup=8000.0),
        ],
        loads=[Load(id="ld", bus="b1", p_mw=50.0, q_mvar=0.0)],
    )
    periods = [Period(load_p_mw={"ld": p}, reserve_mw=reserve_mw) for p in (50.0, 80.0, 50.0)]
    return Scenario(network=network, periods=periods)


uc = MarketMultiperiodOptions(unit_commitment=True)
for label, sc in (("energy only", scenario()), ("plus 60 MW of reserve", scenario(60.0))):
    result = solve_multiperiod(sc, uc)
    schedule = next(c for c in result.commitment if c.id == "baseload")
    on = "".join("1" if x else "0" for x in schedule.on)
    print(f"{label:22s} baseload on={on}  cost {result.objective_cost:6.0f} $  ", end="")
    print("LMP", [round(p.buses[0].lmp) for p in result.periods], "reserve price", [round(p.reserve_price) for p in result.periods])

energy only            baseload on=000  cost   9000 $  LMP [50, 50, 50] reserve price [0, 0, 0]
plus 60 MW of reserve  baseload on=111  cost   9800 $  LMP [10, 10, 10] reserve price [0, 0, 0]


On energy alone the $8000 start-up does not pay for itself and the peaker serves everything at
$50/MWh. A 60 MW reserve requirement the peaker cannot hold alone (it may hold 20) is what
commits the baseload unit: once it is on, reserve is free — the fleet has headroom to spare — so
the reserve price is zero and the energy price falls to $10/MWh. The start-up is recovered
through neither; `commitment_cost` is what an uplift would have to cover, which the manual's
[Unit commitment](../manual/multiperiod.md#unit-commitment) page discusses.

## Where this leaves you

- [N-1 screening](../manual/n1.md) and its [N-2](../manual/n1.md#n-2-branch-pairs) and
  [emergency rating](../manual/n1.md#emergency-ratings) sections, for the screen-then-confirm
  pattern and the agreement guarantees behind it.
- [Security-constrained OPF](../manual/scopf.md), for the constraint-generation loop,
  [corrective redispatch](../manual/scopf.md#corrective-redispatch) and the pricing of a
  binding contingency.
- [Multiperiod market](../manual/multiperiod.md), for commitment,
  [minimum up and down times](../manual/multiperiod.md#minimum-up-and-down-times) and
  [spinning reserve](../manual/multiperiod.md#spinning-reserve).
- [AC-OPF](../manual/ac-opf.md), the one tool here that leaves the DC model: losses, voltages
  and reactive dispatch, with the DC answer as the thing it refines.

Every one of these is also a `jobs` kind, so a service can run it from one JSON request.